# Slow-Burn — a quantitative teardown 🔬
### Decay vs the 0.5·L·(L−1)·σ² formula · Sharpe/drawdown · the regime path-dependence

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Leverage is a free amplifier?: Busted](https://img.shields.io/badge/Leverage_is_a_free_amplifier%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). The drag is real and matches theory; the leverage adds no Sharpe.

> ⚠️ **Not investment advice.** TQQQ vs QQQ, daily total return (Yahoo), 2010–2026. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (slow_burn/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from slow_burn import data, strategy as st
ret = data.fetch_pair()                        # cache-first
g = st.decay_gap(ret["QQQ"], L=3.0)


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **Real** | decay ~8–13%/yr matches 0.5·L·(L−1)·σ² |
| Tradability | **Mirage** | Sharpe 0.90 < 0.98, −82% DD, −79% in 2022 |
| Free amplifier? | **Busted** | more vol/tail, not more return-per-risk |

> 💡 *In plain words:* the drag is arithmetic; the free money is the myth.

## 1 · The claim, steelmanned

- **H₁:** a 3× ETF realizes below 3×(index CAGR) by the volatility drag.
- **H₂:** the drag matches 0.5·L·(L−1)·σ².
- **H₃:** the leverage improves the Sharpe (the 'free amplifier' claim).

## 2 · So what? — what rides on each

H₁/H₂ are the mechanics; H₃ is the investment question. If H₃ fails, the product is risk for risk's sake plus a drag.

## 3 · How we'd know — the protocol

Realized 3×-daily CAGR vs 3×(index CAGR) → the drag vs the formula → Sharpe/drawdown comparison → regime split.

## 4 · The teardown

### 4.1 The drag matches theory

In [2]:
import pandas as pd
print({k:round(v,3) for k,v in g.items()})
print(f"realized decay {g['decay']:+.1%}/yr vs theory {g['drag_theory']:.1%}/yr")

{'underlying_cagr': 0.198, 'naive_Lx_cagr': 0.593, 'levered_cagr': 0.509, 'decay': 0.084, 'drag_theory': 0.128, 'L': 3.0}
realized decay +8.4%/yr vs theory 12.8%/yr


> 💡 *In plain words:* ~8–13%/yr, in line with 0.5·L·(L−1)·σ². **H₁, H₂ hold** — the drag is real.

### 4.2 No Sharpe gain

In [3]:
display(pd.DataFrame({'QQQ':st.summary(ret['QQQ']),'TQQQ':st.summary(ret['TQQQ'])}).T[['cagr','sharpe','vol_ann','max_drawdown']].round(3))

,cagr,sharpe,vol_ann,max_drawdown
QQQ,0.198,0.978,0.206,-0.351
TQQQ,0.437,0.903,0.611,-0.817


> 💡 *In plain words:* TQQQ Sharpe 0.90 < QQQ 0.98, −82% drawdown. **H₃ rejected** — leverage amplified risk, not return-per-risk.

### 4.3 The regime path-dependence

In [4]:
for lab,sl in [('2010-2021 bull',ret['TQQQ'].loc[:'2021']),('2022 bear',ret['TQQQ'].loc['2022']),('2022-on',ret['TQQQ'].loc['2022':])]:
    s=st.summary(sl); print(f'{lab}: CAGR {s["cagr"]:+.0%}, maxDD {s["max_drawdown"]:.0%}')

2010-2021 bull: CAGR +56%, maxDD -70%
2022 bear: CAGR -79%, maxDD -81%
2022-on: CAGR +16%, maxDD -81%


> 💡 *In plain words:* +56%/yr in the bull, −79% in 2022 — it doesn't 'decay to zero', it's a violent regime bet. The drag is the steady cost; the regime is the gamble.

## 5 · The verdict

H₁/H₂ hold, H₃ rejected → Signal `REAL` (the drag), Tradability `MIRAGE`, the free-amplifier claim `BUSTED`.

## 6 · Could you trade it?

Tactically, in a confirmed uptrend, sized for the −80% tail — not as a buy-and-hold. The arithmetic guarantees a drag; the regime decides whether you're up 10× or down 80%.

## 7 · Going further

Forks: (a) a vol-targeted leverage (scale L by inverse vol) — does managing the drag help? (b) 2× vs 3× (the drag scales L·(L−1)); (c) inverse ETFs (SQQQ), where the drag plus a falling-market base is doubly punishing. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).